# 02a — EXP-002: MusiConGen 재생성 파이프라인

Demucs로 분리한 원곡 스템 중 하나(`target_stem`)를 MusiConGen으로 재생성하고, 다시 Demucs로 분리한 뒤 madmom 기반 Alignment Engine으로 원곡과 타이밍을 맞추는 파이프라인.

설정값은 `experiments/exp_002_musicongen/config.yaml` 참고. 파이프라인 전체 구조는 Notion "02번 노트북 — 재생성/재조합 로직 설계" 페이지.

**실행 환경**: Google Colab, GPU 런타임(T4) 필수. MusiConGen 추론에 12GB+ VRAM 권장, T4는 16GB라 충족.

**이번 실행의 목적**: `duration_sec`를 최소(10s)/중간(15s)/최대(20s) 세 구간으로 스윕, `seed`는 42로 전 구간 고정해서 최적 duration_sec 탐색 (2026-08-28 TODO).

## 0. GPU 확인

In [ ]:
!nvidia-smi


## 1. 저장소 클론 + MusiConGen 설치

- `stem-remix-assistant`: 원곡/스템 샘플(`docs/samples/`)과 `config.yaml` 확보용
- `MusiConGen`: 공식 저장소 (https://github.com/YatingMusic/MusiConGen)

In [ ]:
!git clone https://github.com/finneKIM/stem-remix-assistant.git
!git clone https://github.com/YatingMusic/MusiConGen.git

%cd MusiConGen
!pip install -r requirements.txt -q
!conda install -y 'ffmpeg<5' -c conda-forge 2>/dev/null || apt-get -y install ffmpeg -q


## 2. 체크포인트 다운로드

공식 안내대로 HuggingFace(`Cyan0731/MusiConGen`)에서 `compression_state_dict.bin`, `state_dict.bin`을 받아
`audiocraft/ckpt/musicongen/`에 배치. (README 기준 확인된 절차 — 임의 추정 아님)

In [ ]:
from huggingface_hub import hf_hub_download
import os

ckpt_dir = "audiocraft/ckpt/musicongen"
os.makedirs(ckpt_dir, exist_ok=True)

for fname in ["compression_state_dict.bin", "state_dict.bin"]:
    path = hf_hub_download(repo_id="Cyan0731/MusiConGen", filename=fname, local_dir=ckpt_dir)
    print("downloaded:", path)


## 3. 분석 도구 설치 (madmom) + config.yaml 로드

In [ ]:
!pip install madmom pyyaml librosa soundfile -q

import yaml

with open("../stem-remix-assistant/experiments/exp_002_musicongen/config.yaml") as f:
    cfg = yaml.safe_load(f)

TARGET_STEM = cfg["target_stem"]
PROMPT = cfg["prompt"]
DURATION_SWEEP = cfg["duration_sec_sweep"]
SEED = cfg["seed"]

print("target_stem:", TARGET_STEM)
print("prompt:", PROMPT)
print("duration_sec_sweep:", DURATION_SWEEP)
print("seed:", SEED)


## 4. 원곡·원곡 스템에서 BPM/코드/비트 추출

EXP-001에서 만든 `docs/samples/draft_0.wav`(원곡)와 `docs/samples/{target_stem}.wav`(원곡 스템)를 기준으로 madmom 분석. 여기서 얻은 BPM/코드가 MusiConGen 조건화 입력이자, 나중에 재생성 스템과 비교할 기준값.

In [ ]:
from madmom.features.beats import RNNBeatProcessor, DBNBeatTrackingProcessor
from madmom.features.tempo import TempoEstimationProcessor
from madmom.features.chords import DeepChromaChordRecognitionProcessor
from madmom.features.chroma import DeepChromaProcessor
from madmom.features.onsets import CNNOnsetProcessor, OnsetPeakPickingProcessor
import numpy as np

ORIG_TRACK = "../stem-remix-assistant/docs/samples/draft_0.wav"
ORIG_STEM = f"../stem-remix-assistant/docs/samples/{TARGET_STEM}.wav"

def get_bpm(path):
    act = RNNBeatProcessor()(path)
    tempo_proc = TempoEstimationProcessor(fps=100)
    tempi = tempo_proc(act)
    return float(tempi[0][0])  # 가장 confidence 높은 BPM 후보

def get_beats(path):
    act = RNNBeatProcessor()(path)
    beat_proc = DBNBeatTrackingProcessor(fps=100)
    return beat_proc(act)

def get_onsets(path):
    act = CNNOnsetProcessor()(path)
    onset_proc = OnsetPeakPickingProcessor(fps=100)
    return onset_proc(act)

def get_chords(path):
    chroma = DeepChromaProcessor()(path)
    chord_proc = DeepChromaChordRecognitionProcessor()
    return chord_proc(chroma)  # [(start, end, chord_label), ...]

orig_bpm = get_bpm(ORIG_TRACK)
orig_beats = get_beats(ORIG_STEM)
orig_onsets = get_onsets(ORIG_STEM)
orig_chords = get_chords(ORIG_TRACK)

print("원곡 BPM:", round(orig_bpm, 1))
print("원곡 코드 진행(앞 4개):", orig_chords[:4])


## 5. MusiConGen 조건화 입력 준비

madmom이 추출한 코드 라벨(`C:maj`, `A:min` 등)을 MusiConGen의 `generate_with_chords_and_beats` 입력 형식(공백으로 구분된 코드 시퀀스 문자열)으로 변환.

**주의**: MusiConGen 공식 스크립트(`generate_chord_beat.py`)에는 `seed` 파라미터가 없음 — 재현성 확보를 위해 아래처럼 `torch.manual_seed()`를 생성 직전에 직접 호출.

In [ ]:
def chords_to_musicongen_format(chord_events, n_bars=8):
    # (start, end, label) 리스트를 MusiConGen 예제 형식("C G A:min F")에 맞춰
    # 마디 단위로 대표 코드만 뽑아 공백 구분 문자열로 변환.
    # 실제 마디 길이(원곡 BPM/박자 기준)에 맞춰 보정 필요 — 여기서는 단순 등간격 샘플링.
    labels = [c[2] for c in chord_events if c[2] != "N"]
    if not labels:
        labels = ["C"]
    step = max(1, len(labels) // n_bars)
    sampled = labels[::step][:n_bars]
    return " ".join(sampled)

chord_str = chords_to_musicongen_format(orig_chords)
bpm_int = int(round(orig_bpm))
print("MusiConGen 조건화 코드 문자열:", chord_str)
print("MusiConGen 조건화 BPM:", bpm_int)


## 6. MusiConGen 로드 및 duration_sec 스윕 생성

In [ ]:
import torch
import audiocraft
from audiocraft.data.audio import audio_write

musicgen = audiocraft.models.MusicGen.get_pretrained("./ckpt/musicongen")

results_dir = "../stem-remix-assistant/experiments/exp_002_musicongen/outputs"
import os
os.makedirs(results_dir, exist_ok=True)

generated_paths = {}

for duration in DURATION_SWEEP:
    torch.manual_seed(SEED)  # 스윕 전 구간 동일 seed 고정

    musicgen.set_generation_params(duration=duration, extend_stride=duration // 2, top_k=250)

    wav = musicgen.generate_with_chords_and_beats(
        [PROMPT],
        [chord_str],
        [bpm_int],
        [4],  # 4/4 박자 가정
    )

    out_path = f"{results_dir}/musicongen_{TARGET_STEM}_{duration}s_seed{SEED}"
    audio_write(out_path, wav[0].cpu(), musicgen.sample_rate, strategy="loudness", loudness_compressor=True)
    generated_paths[duration] = out_path + ".wav"
    print(f"생성 완료: duration={duration}s -> {out_path}.wav")


## 7. 재생성 트랙 Demucs 재분리 → target_stem만 추출

In [ ]:
!pip install demucs -q

import subprocess

demucs_out_dir = f"{results_dir}/demucs_separated"

extracted_stems = {}
for duration, path in generated_paths.items():
    subprocess.run(
        ["python", "-m", "demucs", "-n", "htdemucs", "-o", demucs_out_dir, path],
        check=True,
    )
    track_name = os.path.splitext(os.path.basename(path))[0]
    stem_path = f"{demucs_out_dir}/htdemucs/{track_name}/{TARGET_STEM}.wav"
    extracted_stems[duration] = stem_path
    print(f"duration={duration}s -> {stem_path}")


## 8. Alignment Engine — Beat Align / Transient Align / Time Stretch

재생성 스템을 원곡 스템 기준으로 보정. 세 단계는 독립적으로 나눠서, 어느 단계에서 얼마나 개선되는지 확인 가능하게 구성.

- **Beat Align**: 재생성 스템 첫 비트를 원곡 스템 첫 비트에 맞춰 앞뒤로 자름(오프셋 보정)
- **Time Stretch**: 재생성 스템 BPM과 원곡 BPM의 비율만큼 librosa로 타임 스트레칭
- **Transient Align**: 온셋 단위 미세 보정 (여기서는 온셋 오프셋 표준편차만 측정, 실제 워핑은 다음 단계 과제로 남김)

In [ ]:
import librosa
import soundfile as sf

def beat_align(gen_path, gen_beats, orig_beats, out_path):
    y, sr = librosa.load(gen_path, sr=None)
    if len(gen_beats) == 0 or len(orig_beats) == 0:
        sf.write(out_path, y, sr)
        return out_path
    offset_sec = gen_beats[0] - orig_beats[0]
    offset_samples = int(offset_sec * sr)
    y_aligned = y[max(0, offset_samples):] if offset_samples > 0 else np.concatenate([np.zeros(-offset_samples), y])
    sf.write(out_path, y_aligned, sr)
    return out_path

def time_stretch_to_bpm(path, current_bpm, target_bpm, out_path):
    y, sr = librosa.load(path, sr=None)
    rate = current_bpm / target_bpm if target_bpm else 1.0
    y_stretched = librosa.effects.time_stretch(y, rate=rate)
    sf.write(out_path, y_stretched, sr)
    return out_path

aligned_stems = {}
alignment_dir = f"{results_dir}/aligned"
os.makedirs(alignment_dir, exist_ok=True)

for duration, stem_path in extracted_stems.items():
    gen_bpm = get_bpm(stem_path)
    gen_beats = get_beats(stem_path)

    stretched_path = f"{alignment_dir}/{TARGET_STEM}_{duration}s_stretched.wav"
    time_stretch_to_bpm(stem_path, gen_bpm, orig_bpm, stretched_path)

    gen_beats_stretched = get_beats(stretched_path)
    final_path = f"{alignment_dir}/{TARGET_STEM}_{duration}s_final.wav"
    beat_align(stretched_path, gen_beats_stretched, orig_beats, final_path)

    aligned_stems[duration] = final_path
    print(f"duration={duration}s: 원본 BPM {round(gen_bpm,1)} -> 보정 후 원곡 BPM {round(orig_bpm,1)}에 정렬")


## 9. 정량 지표 계산 (BPM 오차 / Beat alignment / Onset alignment)

`docs/PROPOSAL.md` 4.3절 평가 방법 기준. 세 duration 후보를 비교해 최적값 후보를 정함.

In [ ]:
import pandas as pd

def beat_alignment_score(beats_a, beats_b):
    # 두 비트 시퀀스를 가까운 것끼리 매칭했을 때의 평균 오차(초) — 작을수록 정렬 잘 됨
    if len(beats_a) == 0 or len(beats_b) == 0:
        return None
    errors = [min(abs(a - b) for b in beats_b) for a in beats_a]
    return float(np.mean(errors))

def onset_alignment_score(onsets_a, onsets_b):
    if len(onsets_a) == 0 or len(onsets_b) == 0:
        return None
    errors = [min(abs(a - b) for b in onsets_b) for a in onsets_a]
    return float(np.mean(errors))

rows = []
for duration, final_path in aligned_stems.items():
    final_bpm = get_bpm(final_path)
    final_beats = get_beats(final_path)
    final_onsets = get_onsets(final_path)

    rows.append({
        "duration_sec": duration,
        "bpm_error": round(abs(final_bpm - orig_bpm), 2),
        "beat_alignment_sec": round(beat_alignment_score(final_beats, orig_beats), 3)
            if beat_alignment_score(final_beats, orig_beats) is not None else None,
        "onset_alignment_sec": round(onset_alignment_score(final_onsets, orig_onsets), 3)
            if onset_alignment_score(final_onsets, orig_onsets) is not None else None,
    })

results_df = pd.DataFrame(rows).sort_values("duration_sec")
results_df.to_csv(f"{results_dir}/duration_sweep_results.csv", index=False)
results_df


## 10. 다음 작업

1. 위 표에서 `bpm_error` + `beat_alignment_sec` + `onset_alignment_sec`가 가장 낮은 `duration_sec`를 최적값으로 선택
2. `experiments/exp_002_musicongen/config.yaml`의 `duration_sec`에 확정값 기록, `duration_sec_sweep` 필드는 그대로 이력으로 남김
3. 청취 평가(사람이 직접 듣고 자연스러움 판단) 진행 — `docs/experiments/model_comparison.md`에 기록
4. `experiments/exp_002_musicongen/README.md`의 "결과"·"결론" 섹션 채우기
5. 동일한 duration_sec/seed 스윕 설계를 EXP-003(MusicGen-Melody/Style)에도 적용해 공정 비교 조건 맞추기